# 3-D Heat Conduction: Cartesian and Spherical Coordinates

## ChBE 3300: Multidimensional Fluids and Heat Transport

### Overview

This notebook covers two important cases of 3-D heat conduction relevant to chemical and biomolecular engineering:

1. **Steady-State Cartesian**: Heat generation in a catalytic reactor block (Petrochemical)
2. **Transient Spherical**: Thermal treatment of a spherical bioreactor (Biomolecular)

### Learning Objectives
- Understand 3-D heat equation in different coordinate systems
- Implement finite difference methods for steady and unsteady conduction
- Visualize temperature distributions in 3-D
- Apply results to engineering design problems in petrochemical and biomolecular contexts

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Set style
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully!")

---
## Part 1: Steady-State 3D Cartesian Coordinates

### Application: Catalytic Reactor Block with Internal Heat Generation

Consider a cubic block of porous catalyst material used in a fixed-bed reactor for an exothermic petrochemical process (e.g., Fischer-Tropsch synthesis or methanol production):

- **Dimensions**: 0.2 m × 0.2 m × 0.2 m cubic catalyst block
- **Heat Generation**: Uniform volumetric heat generation from exothermic reactions
- **Boundary Conditions**: 
  - Bottom face (z=0): Hot reactant inlet at 250°C
  - Top face (z=L): Product outlet with cooling, 180°C
  - Side faces (x=0, x=L, y=0, y=L): Heat transfer to cooling jacket at 150°C

### Governing Equation

For steady-state 3-D conduction with uniform heat generation:

$$\frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2} + \frac{\partial^2 T}{\partial z^2} + \frac{q_{gen}}{k} = 0$$

where:
- $q_{gen}$ is the volumetric heat generation rate (W/m³)
- $k$ is the effective thermal conductivity of the catalyst bed (W/m·K)

This is the Poisson equation with a source term.

### Finite Difference Discretization

For uniform grid spacing ($\Delta x = \Delta y = \Delta z = \Delta h$):

$$\frac{T_{i+1,j,k} - 2T_{i,j,k} + T_{i-1,j,k}}{\Delta h^2} + \frac{T_{i,j+1,k} - 2T_{i,j,k} + T_{i,j-1,k}}{\Delta h^2} + \frac{T_{i,j,k+1} - 2T_{i,j,k} + T_{i,j,k-1}}{\Delta h^2} + \frac{q_{gen}}{k} = 0$$

Solving for $T_{i,j,k}$:

$$T_{i,j,k} = \frac{1}{6}\left(T_{i+1,j,k} + T_{i-1,j,k} + T_{i,j+1,k} + T_{i,j-1,k} + T_{i,j,k+1} + T_{i,j,k-1} - \frac{q_{gen} \Delta h^2}{k}\right)$$

In [ ]:
# Problem parameters - Catalytic Reactor Block
L = 0.2  # Cube side length (m)
k_cat = 2.5  # Effective thermal conductivity of catalyst bed (W/m·K)
q_gen = 8e5  # Volumetric heat generation from exothermic reaction (W/m³)

# Boundary conditions
T_bottom = 250.0   # Bottom face - hot reactant inlet (°C)
T_top = 180.0      # Top face - product outlet with cooling (°C)
T_sides = 150.0    # Side faces - cooling jacket temperature (°C)

# Grid setup
nx = ny = nz = 30  # Number of points in each direction
dx = dy = dz = L / (nx - 1)

x = np.linspace(0, L, nx)
y = np.linspace(0, L, ny)
z = np.linspace(0, L, nz)

# Initialize 3D temperature field
T = np.zeros((nx, ny, nz))

# Apply boundary conditions
T[:, :, 0] = T_bottom      # Bottom (z=0)
T[:, :, -1] = T_top        # Top (z=L)
T[0, :, :] = T_sides       # Left (x=0)
T[-1, :, :] = T_sides      # Right (x=L)
T[:, 0, :] = T_sides       # Front (y=0)
T[:, -1, :] = T_sides      # Back (y=L)

# Initial guess for interior (average of boundary conditions + heat generation effect)
T_avg = (T_bottom + T_top + 4*T_sides) / 6
T[1:-1, 1:-1, 1:-1] = T_avg + 20  # Add offset for heat generation

print("Catalytic Reactor Block Problem Setup:")
print(f"  Block dimensions: {L*100:.1f} cm × {L*100:.1f} cm × {L*100:.1f} cm")
print(f"  Grid: {nx} × {ny} × {nz} = {nx*ny*nz:,} points")
print(f"  Thermal conductivity: {k_cat} W/m·K")
print(f"  Heat generation rate: {q_gen/1e6:.2f} MW/m³")
print(f"  Total heat generation: {q_gen * L**3 / 1000:.2f} kW")

In [ ]:
# Solve using Gauss-Seidel iteration
max_iter = 5000
tolerance = 1e-4
source_term = -q_gen * dx**2 / k_cat  # Source term for finite difference

print("Solving steady-state 3D Cartesian problem...")
print("This may take a minute...\n")

for iteration in range(max_iter):
    T_old = T.copy()
    
    # Update interior points
    for i in range(1, nx-1):
        for j in range(1, ny-1):
            for k in range(1, nz-1):
                T[i, j, k] = (T[i+1, j, k] + T[i-1, j, k] + 
                             T[i, j+1, k] + T[i, j-1, k] +
                             T[i, j, k+1] + T[i, j, k-1] + source_term) / 6.0
    
    # Maintain boundary conditions
    T[:, :, 0] = T_bottom
    T[:, :, -1] = T_top
    T[0, :, :] = T_sides
    T[-1, :, :] = T_sides
    T[:, 0, :] = T_sides
    T[:, -1, :] = T_sides
    
    # Check convergence
    error = np.max(np.abs(T - T_old))
    
    if iteration % 500 == 0:
        print(f"  Iteration {iteration:5d}: error = {error:.6e}, T_max = {T.max():.2f}°C")
    
    if error < tolerance:
        print(f"\n✓ Converged after {iteration} iterations")
        print(f"  Final error: {error:.6e}")
        break

# Calculate maximum temperature and location
T_max = T.max()
max_idx = np.unravel_index(T.argmax(), T.shape)
x_max, y_max, z_max = x[max_idx[0]], y[max_idx[1]], z[max_idx[2]]

print(f"\nTemperature Analysis:")
print(f"  Maximum temperature: {T_max:.1f}°C")
print(f"  Location: x={x_max*100:.1f} cm, y={y_max*100:.1f} cm, z={z_max*100:.1f} cm")
print(f"  Minimum temperature: {T.min():.1f}°C")
print(f"  Center temperature: {T[nx//2, ny//2, nz//2]:.1f}°C")

# Calculate heat removal at boundaries
q_bottom = -k_cat * np.mean((T[:, :, 1] - T[:, :, 0]) / dz)
q_top = -k_cat * np.mean((T[:, :, -1] - T[:, :, -2]) / dz)
q_side_x0 = -k_cat * np.mean((T[1, :, :] - T[0, :, :]) / dx)
q_side_xL = -k_cat * np.mean((T[-1, :, :] - T[-2, :, :]) / dx)

Q_bottom = q_bottom * L * L
Q_top = q_top * L * L
Q_sides = 2 * (q_side_x0 + q_side_xL) * L * L
Q_total = Q_bottom + Q_top + Q_sides

print(f"\nHeat Removal Rates:")
print(f"  Bottom face: {Q_bottom/1000:.2f} kW")
print(f"  Top face: {Q_top/1000:.2f} kW")
print(f"  Side faces (total): {Q_sides/1000:.2f} kW")
print(f"  Total heat removal: {Q_total/1000:.2f} kW")
print(f"  Heat generated: {q_gen * L**3 / 1000:.2f} kW")
print(f"  Energy balance error: {abs(Q_total - q_gen*L**3)/(q_gen*L**3)*100:.2f}%")

In [ ]:
# Visualization - Catalytic Reactor Block
fig = plt.figure(figsize=(20, 14))

# 1. Temperature slices at different z-levels
z_slices = [0, nz//4, nz//2, 3*nz//4, -1]
z_labels = ['Bottom (z=0)', 'z=L/4', 'Center (z=L/2)', 'z=3L/4', 'Top (z=L)']

for idx, (z_idx, label) in enumerate(zip(z_slices, z_labels)):
    ax = plt.subplot(3, 5, idx + 1)
    X_slice, Y_slice = np.meshgrid(x*100, y*100)
    contourf = ax.contourf(X_slice, Y_slice, T[:, :, z_idx].T, levels=20, cmap='hot')
    contour = ax.contour(X_slice, Y_slice, T[:, :, z_idx].T, levels=8, colors='black', 
                         alpha=0.3, linewidths=0.5)
    ax.clabel(contour, inline=True, fontsize=7, fmt='%0.0f°C')
    plt.colorbar(contourf, ax=ax, label='T (°C)')
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')
    ax.set_title(label)
    ax.set_aspect('equal')
    # Mark maximum if it's on this slice
    if z_idx == max_idx[2]:
        ax.plot(x_max*100, y_max*100, 'w*', markersize=15, 
                markeredgecolor='red', markeredgewidth=2)

# 2. Temperature profiles along centerlines
ax6 = plt.subplot(3, 5, 6)
ax6.plot(x*100, T[:, ny//2, nz//2], 'r-', linewidth=2.5, label='Along x-axis')
ax6.plot(y*100, T[nx//2, :, nz//2], 'b-', linewidth=2.5, label='Along y-axis')
ax6.plot(z*100, T[nx//2, ny//2, :], 'g-', linewidth=2.5, label='Along z-axis')
ax6.set_xlabel('Position (cm)')
ax6.set_ylabel('Temperature (°C)')
ax6.set_title('Temperature Profiles Through Center')
ax6.grid(True, alpha=0.3)
ax6.legend()

# 3. Temperature distribution histogram
ax7 = plt.subplot(3, 5, 7)
ax7.hist(T.flatten(), bins=50, color='orange', edgecolor='black', alpha=0.7)
ax7.axvline(T_max, color='r', linestyle='--', linewidth=2, label=f'Max: {T_max:.1f}°C')
ax7.axvline(T[nx//2, ny//2, nz//2], color='b', linestyle='--', linewidth=2, 
            label=f'Center: {T[nx//2, ny//2, nz//2]:.1f}°C')
ax7.set_xlabel('Temperature (°C)')
ax7.set_ylabel('Frequency')
ax7.set_title('Temperature Distribution')
ax7.legend()
ax7.grid(True, alpha=0.3)

# 4. Vertical slice through center (y-z plane)
ax8 = plt.subplot(3, 5, 8)
Y_vert, Z_vert = np.meshgrid(y*100, z*100)
contourf8 = ax8.contourf(Y_vert, Z_vert, T[nx//2, :, :].T, levels=20, cmap='hot')
plt.colorbar(contourf8, ax=ax8, label='T (°C)')
ax8.set_xlabel('y (cm)')
ax8.set_ylabel('z (cm)')
ax8.set_title('Vertical Slice (x=L/2)')
ax8.set_aspect('equal')

# 5. Vertical slice through center (x-z plane)
ax9 = plt.subplot(3, 5, 9)
X_vert, Z_vert = np.meshgrid(x*100, z*100)
contourf9 = ax9.contourf(X_vert, Z_vert, T[:, ny//2, :].T, levels=20, cmap='hot')
plt.colorbar(contourf9, ax=ax9, label='T (°C)')
ax9.set_xlabel('x (cm)')
ax9.set_ylabel('z (cm)')
ax9.set_title('Vertical Slice (y=L/2)')
ax9.set_aspect('equal')

# 6. Temperature gradient magnitude at center slice
ax10 = plt.subplot(3, 5, 10)
# Calculate gradient magnitude at z=L/2
z_center = nz//2
dTdx = np.gradient(T[:, :, z_center], dx, axis=0)
dTdy = np.gradient(T[:, :, z_center], dy, axis=1)
grad_mag = np.sqrt(dTdx**2 + dTdy**2)
contourf10 = ax10.contourf(X_slice, Y_slice, grad_mag.T, levels=20, cmap='viridis')
plt.colorbar(contourf10, ax=ax10, label='|∇T| (°C/m)')
ax10.set_xlabel('x (cm)')
ax10.set_ylabel('y (cm)')
ax10.set_title('Temperature Gradient Magnitude (z=L/2)')
ax10.set_aspect('equal')

# 7-10. 3D visualizations with isosurfaces
# Create isosurface data
T_levels = [200, 220, 240, 260]
for idx, T_level in enumerate(T_levels):
    ax = fig.add_subplot(3, 5, 11 + idx, projection='3d')
    
    # Find points near this temperature
    mask = np.abs(T - T_level) < 5
    pts = np.where(mask)
    
    if len(pts[0]) > 0:
        ax.scatter(x[pts[0]]*100, y[pts[1]]*100, z[pts[2]]*100, 
                  c=T[mask], cmap='hot', alpha=0.3, s=1)
    
    # Draw box outline
    corners = np.array([[0, 0, 0], [L, 0, 0], [L, L, 0], [0, L, 0], [0, 0, 0],
                        [0, 0, L], [L, 0, L], [L, L, L], [0, L, L], [0, 0, L]]) * 100
    ax.plot(corners[:5, 0], corners[:5, 1], corners[:5, 2], 'k-', alpha=0.3)
    ax.plot(corners[5:, 0], corners[5:, 1], corners[5:, 2], 'k-', alpha=0.3)
    for i in range(4):
        ax.plot([corners[i, 0], corners[i+5, 0]], 
               [corners[i, 1], corners[i+5, 1]], 
               [corners[i, 2], corners[i+5, 2]], 'k-', alpha=0.3)
    
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')
    ax.set_zlabel('z (cm)')
    ax.set_title(f'Isosurface: T ≈ {T_level}°C')
    ax.view_init(elev=20, azim=45)

# 15. 3D scatter plot of temperature
ax15 = fig.add_subplot(3, 5, 15, projection='3d')
# Sample every other point for visibility
skip = 3
X3d, Y3d, Z3d = np.meshgrid(x[::skip], y[::skip], z[::skip], indexing='ij')
T_sample = T[::skip, ::skip, ::skip]
scatter = ax15.scatter(X3d.flatten()*100, Y3d.flatten()*100, Z3d.flatten()*100,
                      c=T_sample.flatten(), cmap='hot', alpha=0.4, s=20)
plt.colorbar(scatter, ax=ax15, label='T (°C)', shrink=0.5)
ax15.set_xlabel('x (cm)')
ax15.set_ylabel('y (cm)')
ax15.set_zlabel('z (cm)')
ax15.set_title('3D Temperature Field')
ax15.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig('../../figures/reactor_block_steady_3d_cartesian.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/reactor_block_steady_3d_cartesian.png")

### Engineering Analysis - Catalytic Reactor

**Key Insights:**

1. **Hot Spot Formation**: The maximum temperature occurs in the interior, not at boundaries
   - Location depends on heat generation rate and boundary cooling
   - Critical for catalyst deactivation and selectivity
   - Must remain below maximum allowable temperature (~300-400°C for most catalysts)

2. **Three-Dimensional Effects**:
   - Temperature varies in all three directions
   - Corner and edge regions are cooler due to multi-directional heat flow
   - 1-D models would significantly underestimate maximum temperature

3. **Heat Removal Strategy**:
   - Multiple cooling boundaries needed for effective temperature control
   - Side cooling (jacket) removes significant heat
   - Top and bottom faces have different heat fluxes due to different boundary temperatures

4. **Design Considerations**:
   - **Catalyst activity**: Higher at higher temperatures, but limited by deactivation
   - **Selectivity**: Temperature affects product distribution in competing reactions
   - **Runaway prevention**: Heat generation rate must not exceed heat removal capacity
   - **Scale-up**: Larger reactors have lower surface-to-volume ratio, making cooling more challenging

5. **Optimization Opportunities**:
   - Adjust cooling temperatures on different faces
   - Non-uniform catalyst loading (less active catalyst in hot zones)
   - Internal cooling channels
   - Optimal reactor aspect ratio

**Industrial Applications**:
- Fischer-Tropsch synthesis (gas-to-liquids)
- Methanol synthesis
- Ammonia synthesis
- Selective catalytic reduction (SCR)
- Catalytic cracking

---
## Part 2: Transient 3D Spherical Coordinates

### Application: Thermal Sterilization of a Spherical Bioreactor

Consider a spherical bioreactor vessel containing mammalian cell culture that needs to be thermally sterilized before use:

- **Geometry**: Spherical vessel, radius = 0.15 m (30 cm diameter)
- **Contents**: Cell culture medium with suspended cells
- **Initial Condition**: Uniform temperature at 25°C (room temperature)
- **Process**: Surface suddenly exposed to steam at 121°C for sterilization
- **Objective**: Ensure entire volume reaches 110°C for adequate sterilization while minimizing thermal stress on vessel materials

### Governing Equation

For transient 3-D conduction in spherical coordinates with radial symmetry (no θ or φ dependence):

$$\frac{\partial T}{\partial t} = \alpha \left[\frac{1}{r^2}\frac{\partial}{\partial r}\left(r^2\frac{\partial T}{\partial r}\right)\right]$$

Expanding:

$$\frac{\partial T}{\partial t} = \alpha \left[\frac{\partial^2 T}{\partial r^2} + \frac{2}{r}\frac{\partial T}{\partial r}\right]$$

where:
- $\alpha = k/(\rho c_p)$ is thermal diffusivity (m²/s)
- $r$ is radial distance from center (m)

### Finite Difference Discretization (Explicit Method)

Using central differences:

$$\frac{T_i^{n+1} - T_i^n}{\Delta t} = \alpha \left[\frac{T_{i+1}^n - 2T_i^n + T_{i-1}^n}{\Delta r^2} + \frac{2}{r_i}\frac{T_{i+1}^n - T_{i-1}^n}{2\Delta r}\right]$$

For $i=0$ (center), use L'Hôpital's rule:

$$\lim_{r \to 0} \left[\frac{\partial^2 T}{\partial r^2} + \frac{2}{r}\frac{\partial T}{\partial r}\right] = 3\frac{\partial^2 T}{\partial r^2}$$

### Stability Criterion

For spherical coordinates:

$$\Delta t \leq \frac{\Delta r^2}{6\alpha}$$

In [ ]:
# Problem parameters - Spherical Bioreactor
R_sphere = 0.15  # Sphere radius (m) - 30 cm diameter bioreactor

# Thermal properties of cell culture medium (similar to water)
k_bio = 0.6        # Thermal conductivity (W/m·K)
rho_bio = 1020.0   # Density (kg/m³) - slightly higher than water due to dissolved proteins
cp_bio = 3950.0    # Specific heat (J/kg·K)
alpha_bio = k_bio / (rho_bio * cp_bio)  # Thermal diffusivity (m²/s)

# Temperature conditions
T_init_bio = 25.0    # Initial temperature (°C)
T_steam = 121.0      # Steam sterilization temperature (°C)
T_target_bio = 110.0 # Target sterilization temperature (°C)

# Grid setup (radial only due to symmetry)
nr_bio = 100
r_bio = np.linspace(0, R_sphere, nr_bio)
dr_bio = r_bio[1] - r_bio[0]

# Time step (stability criterion for spherical)
dt_max_bio = dr_bio**2 / (6 * alpha_bio)
dt_bio = 0.4 * dt_max_bio  # Use 40% of maximum for safety
total_time_bio = 2400.0  # 40 minutes simulation
nt_bio = int(total_time_bio / dt_bio)

print("Spherical Bioreactor Sterilization Problem Setup:")
print(f"  Reactor radius: {R_sphere*100:.1f} cm (diameter: {2*R_sphere*100:.1f} cm)")
print(f"  Volume: {4/3*np.pi*R_sphere**3*1000:.2f} liters")
print(f"  Thermal diffusivity: {alpha_bio:.3e} m²/s")
print(f"  Radial grid points: {nr_bio}")
print(f"  Maximum stable time step: {dt_max_bio:.3f} s")
print(f"  Using time step: {dt_bio:.3f} s")
print(f"  Number of time steps: {nt_bio:,}")
print(f"  Total simulation time: {total_time_bio} s ({total_time_bio/60:.1f} min)")

# Initialize temperature field
T_bio = np.ones(nr_bio) * T_init_bio

# Storage for analysis
center_temp_bio_history = []
surface_temp_bio_history = []
time_bio_history = []
T_profiles = []  # Store radial profiles at different times
profile_times = [0, 120, 300, 600, 1200, 1800, 2400]  # seconds

In [ ]:
# Solve transient spherical problem
print("\nSolving transient spherical problem...")
print("Progress: ", end='')

for n in range(nt_bio):
    T_old = T_bio.copy()
    current_time = n * dt_bio
    
    # Update interior points (i = 1 to nr-2)
    for i in range(1, nr_bio-1):
        r_i = r_bio[i]
        
        # Second derivative
        d2T_dr2 = (T_old[i+1] - 2*T_old[i] + T_old[i-1]) / dr_bio**2
        
        # First derivative (central difference)
        dT_dr = (T_old[i+1] - T_old[i-1]) / (2 * dr_bio)
        
        # Update temperature
        T_bio[i] = T_old[i] + alpha_bio * dt_bio * (d2T_dr2 + (2/r_i) * dT_dr)
    
    # Special treatment at center (r=0) using L'Hôpital's rule
    # At r=0: dT/dt = 3α * d²T/dr²
    d2T_dr2_center = (T_old[1] - T_old[0]) / dr_bio**2  # One-sided difference
    T_bio[0] = T_old[0] + 3 * alpha_bio * dt_bio * d2T_dr2_center
    
    # Apply boundary condition at surface
    T_bio[-1] = T_steam
    
    # Record data
    if n % 50 == 0:
        center_temp_bio_history.append(T_bio[0])
        surface_temp_bio_history.append(T_bio[-1])
        time_bio_history.append(current_time)
    
    # Save profiles at specific times
    if current_time in profile_times or (len(T_profiles) < len(profile_times) and 
                                         current_time >= profile_times[len(T_profiles)]):
        if len(T_profiles) < len(profile_times):
            T_profiles.append(T_bio.copy())
            print(f"\n  t = {current_time:.0f} s ({current_time/60:.1f} min): T_center = {T_bio[0]:.1f}°C", end='')
    
    # Progress indicator
    if n % (nt_bio // 20) == 0:
        print('.', end='', flush=True)
    
    # Check if target temperature reached at center
    if T_bio[0] >= T_target_bio and len(center_temp_bio_history) > 1:
        if center_temp_bio_history[-2] < T_target_bio:
            time_to_sterilize = current_time
            print(f"\n\n✓ Sterilization temperature {T_target_bio}°C reached at center after {time_to_sterilize:.1f} s ({time_to_sterilize/60:.2f} min)")

print(f"\n\n✓ Simulation complete!")
print(f"  Final center temperature: {T_bio[0]:.1f}°C")
print(f"  Final minimum temperature: {T_bio.min():.1f}°C")

# Calculate total energy absorbed
Q_absorbed = rho_bio * cp_bio * (4/3 * np.pi * R_sphere**3) * (T_bio.mean() - T_init_bio)
print(f"\nEnergy Analysis:")
print(f"  Total energy absorbed: {Q_absorbed/1e6:.2f} MJ")
print(f"  Average temperature rise: {T_bio.mean() - T_init_bio:.1f}°C")

In [ ]:
# Visualization - Spherical Bioreactor
fig = plt.figure(figsize=(20, 14))

# 1. Radial temperature profiles at different times
ax1 = plt.subplot(3, 4, 1)
colors = plt.cm.plasma(np.linspace(0, 1, len(T_profiles)))
for idx, (T_prof, t) in enumerate(zip(T_profiles, profile_times[:len(T_profiles)])):
    ax1.plot(r_bio*100, T_prof, linewidth=2.5, color=colors[idx], 
            label=f't = {t} s ({t/60:.1f} min)')
ax1.axhline(y=T_target_bio, color='green', linestyle='--', linewidth=2, 
           label=f'Target: {T_target_bio}°C')
ax1.axhline(y=T_steam, color='red', linestyle='--', linewidth=1.5, alpha=0.5,
           label=f'Steam: {T_steam}°C')
ax1.set_xlabel('Radial Position (cm)')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Radial Temperature Profiles')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower right', fontsize=8)

# 2. Center temperature history
ax2 = plt.subplot(3, 4, 2)
ax2.plot(np.array(time_bio_history)/60, center_temp_bio_history, 
        'b-', linewidth=2.5, label='Center')
ax2.axhline(y=T_target_bio, color='g', linestyle='--', linewidth=2,
           label=f'Sterilization: {T_target_bio}°C')
ax2.axhline(y=T_steam, color='r', linestyle='--', linewidth=1.5, alpha=0.5,
           label=f'Steam: {T_steam}°C')
ax2.fill_between(np.array(time_bio_history)/60, T_target_bio, T_steam,
                alpha=0.2, color='green', label='Sterilization zone')
ax2.set_xlabel('Time (min)')
ax2.set_ylabel('Temperature (°C)')
ax2.set_title('Center Temperature vs Time')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='lower right', fontsize=9)

# 3. Temperature gradient at different times
ax3 = plt.subplot(3, 4, 3)
for idx, (T_prof, t) in enumerate(zip(T_profiles[1::2], profile_times[1::2])):
    dT_dr = np.gradient(T_prof, dr_bio)
    ax3.plot(r_bio*100, dT_dr, linewidth=2, label=f't = {t} s')
ax3.set_xlabel('Radial Position (cm)')
ax3.set_ylabel('Temperature Gradient (°C/m)')
ax3.set_title('Radial Temperature Gradient')
ax3.grid(True, alpha=0.3)
ax3.legend(fontsize=8)

# 4. Heat flux at surface vs time
ax4 = plt.subplot(3, 4, 4)
# Calculate heat flux at surface for stored profiles
heat_flux_surface = []
for T_prof in T_profiles:
    q_surf = -k_bio * (T_prof[-1] - T_prof[-2]) / dr_bio
    heat_flux_surface.append(q_surf / 1000)  # Convert to kW/m²
ax4.plot(np.array(profile_times[:len(heat_flux_surface)])/60, heat_flux_surface,
        'ro-', linewidth=2.5, markersize=8)
ax4.set_xlabel('Time (min)')
ax4.set_ylabel('Surface Heat Flux (kW/m²)')
ax4.set_title('Heat Flux at Sphere Surface')
ax4.grid(True, alpha=0.3)

# 5-8. 3D representations of sphere at different times
for plot_idx, (T_prof, t) in enumerate(zip(T_profiles[1:5], profile_times[1:5])):
    ax = fig.add_subplot(3, 4, 5 + plot_idx, projection='3d')
    
    # Create spherical mesh for visualization
    phi = np.linspace(0, np.pi, 30)
    theta = np.linspace(0, 2*np.pi, 40)
    Phi, Theta = np.meshgrid(phi, theta)
    
    # Use surface temperature for coloring
    X_sphere = R_sphere * np.sin(Phi) * np.cos(Theta) * 100
    Y_sphere = R_sphere * np.sin(Phi) * np.sin(Theta) * 100
    Z_sphere = R_sphere * np.cos(Phi) * 100
    
    # Color by surface temperature
    surf_color = np.ones_like(X_sphere) * T_prof[-1]
    
    surf = ax.plot_surface(X_sphere, Y_sphere, Z_sphere, 
                          facecolors=plt.cm.hot((surf_color - T_init_bio)/(T_steam - T_init_bio)),
                          alpha=0.7, antialiased=True)
    
    # Add cross-section
    r_section = np.linspace(0, R_sphere, 20)
    theta_section = np.linspace(0, 2*np.pi, 40)
    R_section, Theta_section = np.meshgrid(r_section, theta_section)
    
    X_section = R_section * np.cos(Theta_section) * 100
    Y_section = R_section * np.sin(Theta_section) * 100
    Z_section = np.zeros_like(X_section)
    
    # Interpolate temperature for cross-section
    T_section_vals = np.interp(R_section.flatten(), r_bio, T_prof).reshape(R_section.shape)
    
    ax.plot_surface(X_section, Y_section, Z_section,
                   facecolors=plt.cm.hot((T_section_vals - T_init_bio)/(T_steam - T_init_bio)),
                   alpha=0.9, antialiased=True)
    
    ax.set_xlabel('x (cm)')
    ax.set_ylabel('y (cm)')
    ax.set_zlabel('z (cm)')
    ax.set_title(f't = {t} s ({t/60:.1f} min)\nT_center = {T_prof[0]:.1f}°C', fontsize=9)
    ax.view_init(elev=20, azim=45)
    
    # Equal aspect ratio
    max_range = R_sphere * 100
    ax.set_xlim([-max_range, max_range])
    ax.set_ylim([-max_range, max_range])
    ax.set_zlim([-max_range, max_range])

# 9. Average temperature vs time
ax9 = plt.subplot(3, 4, 9)
avg_temps = []
for T_prof in T_profiles:
    # Volume-weighted average
    dV = 4 * np.pi * r_bio**2 * dr_bio
    avg_T = np.sum(T_prof * dV) / np.sum(dV)
    avg_temps.append(avg_T)
ax9.plot(np.array(profile_times[:len(avg_temps)])/60, avg_temps,
        'mo-', linewidth=2.5, markersize=8, label='Volume-averaged')
ax9.plot(np.array(profile_times[:len(T_profiles)])/60, 
        [T_prof[0] for T_prof in T_profiles],
        'bs-', linewidth=2, markersize=6, label='Center')
ax9.axhline(y=T_target_bio, color='g', linestyle='--', linewidth=2)
ax9.set_xlabel('Time (min)')
ax9.set_ylabel('Temperature (°C)')
ax9.set_title('Average and Center Temperature')
ax9.grid(True, alpha=0.3)
ax9.legend()

# 10. Penetration depth (distance from surface to target temp)
ax10 = plt.subplot(3, 4, 10)
penetration_depths = []
for T_prof in T_profiles:
    # Find where temperature drops below target
    idx = np.where(T_prof >= T_target_bio)[0]
    if len(idx) > 0:
        r_target = r_bio[idx[0]]
        penetration = R_sphere - r_target
    else:
        penetration = 0
    penetration_depths.append(penetration * 100)
ax10.plot(np.array(profile_times[:len(penetration_depths)])/60, penetration_depths,
         'g^-', linewidth=2.5, markersize=10)
ax10.axhline(y=R_sphere*100, color='r', linestyle='--', linewidth=2,
            label=f'Full radius ({R_sphere*100:.0f} cm)')
ax10.set_xlabel('Time (min)')
ax10.set_ylabel('Penetration Depth (cm)')
ax10.set_title('Sterilization Penetration Depth')
ax10.grid(True, alpha=0.3)
ax10.legend()

# 11. Temperature contour (r-theta plane)
ax11 = plt.subplot(3, 4, 11, projection='polar')
# Use final temperature profile
theta_contour = np.linspace(0, 2*np.pi, 100)
R_contour, Theta_contour = np.meshgrid(r_bio, theta_contour)
T_contour = np.tile(T_profiles[-1], (len(theta_contour), 1))
contourf11 = ax11.contourf(Theta_contour, R_contour*100, T_contour, levels=20, cmap='hot')
plt.colorbar(contourf11, ax=ax11, label='T (°C)', shrink=0.6)
ax11.set_title(f'Final Temperature Distribution\n(t = {profile_times[-1]/60:.0f} min)', 
              fontsize=10, pad=20)
ax11.set_ylabel('Radius (cm)', labelpad=30)

# 12. Energy absorbed vs time
ax12 = plt.subplot(3, 4, 12)
energy_absorbed = []
for T_prof in T_profiles:
    # Calculate total energy
    dV = 4 * np.pi * r_bio**2 * dr_bio
    avg_T = np.sum(T_prof * dV) / np.sum(dV)
    Q = rho_bio * cp_bio * (4/3 * np.pi * R_sphere**3) * (avg_T - T_init_bio)
    energy_absorbed.append(Q / 1e6)  # MJ
ax12.plot(np.array(profile_times[:len(energy_absorbed)])/60, energy_absorbed,
         'rs-', linewidth=2.5, markersize=8)
ax12.set_xlabel('Time (min)')
ax12.set_ylabel('Energy Absorbed (MJ)')
ax12.set_title('Total Energy Absorbed')
ax12.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/bioreactor_sterilization_transient_spherical.png', 
           dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Figure saved to figures/bioreactor_sterilization_transient_spherical.png")

### Engineering Analysis - Bioreactor Sterilization

**Key Insights:**

1. **Spherical Symmetry Advantages**:
   - Only radial variation needs to be considered
   - Uniform heat exposure from all directions
   - Optimal geometry for minimizing temperature gradients
   - Lowest surface area to volume ratio reduces heat loss

2. **Cold Point Location**:
   - Center of sphere is the slowest to heat
   - Must ensure center reaches target temperature for complete sterilization
   - Penetration depth increases with time following ~√(αt) relationship

3. **Processing Time Considerations**:
   - Larger vessels require significantly longer sterilization times (R² scaling)
   - Trade-off between sterilization assurance and energy consumption
   - Typical process: 20-30 minutes at 121°C for complete sterilization

4. **Temperature Gradients and Thermal Stress**:
   - Large temperature gradients induce thermal stress in vessel walls
   - Critical for glass or specialized polymer bioreactors
   - Gradual heating/cooling cycles may be needed for fragile materials
   - Maximum gradient occurs at surface during initial heating

5. **Energy Requirements**:
   - Substantial energy needed to heat large volumes
   - Heat losses through vessel walls (not modeled here but important in practice)
   - Recovery time between batches affects productivity

**Biomolecular Engineering Applications**:

1. **Bioreactor Sterilization**:
   - In-place sterilization (SIP) of fermentation vessels
   - Critical for preventing contamination in cell culture
   - Must reach 121°C for 15-30 minutes (F₀ value calculation)

2. **Cell Culture Media Preparation**:
   - Autoclaving of growth media
   - Heat-sensitive components added post-sterilization

3. **Organoid and Tissue Culture**:
   - Sterilization of spherical culture vessels
   - 3D cell aggregates and organoids

4. **Cryopreservation** (reverse process):
   - Cooling of spherical cell samples
   - Similar mathematics but heat removal instead of addition
   - Critical for preventing ice crystal formation

5. **Scale-Up Considerations**:
   - Lab scale (1-10 L): Fast sterilization (~10-15 min)
   - Pilot scale (100-1000 L): Moderate time (~20-30 min)
   - Production scale (>10,000 L): Extended time (>45 min)
   - Large vessels may need internal heating coils for efficiency

**Optimization Strategies**:

1. **Temperature Ramping**:
   - Gradual temperature increase to reduce thermal stress
   - Controlled heating rate (e.g., 1-2°C/min)

2. **Agitation**:
   - Mixing during heating improves uniformity (convection)
   - Reduces reliance on pure conduction
   - Not modeled here but significant in practice

3. **Pressure Control**:
   - Steam sterilization under pressure (autoclave)
   - 121°C requires ~15 psig (1 bar gauge)

4. **Alternative Sterilization**:
   - Chemical sterilization for heat-sensitive equipment
   - Filtration for liquids
   - UV or gamma irradiation for some applications

5. **Validation**:
   - Use of biological indicators (spore strips)
   - Temperature probes at cold point
   - F₀ value calculation for equivalent lethality

---
## Summary and Comparison

### Coordinate System Selection

| Feature | Cartesian (x,y,z) | Spherical (r,θ,φ) |
|---------|------------------|-------------------|
| **Best for** | Rectangular geometries | Spheres, drops, particles |
| **Complexity** | Moderate | Higher (with symmetry: low) |
| **Grid points** | nx × ny × nz | nr (if symmetric) |
| **Computation** | More points needed | Fewer with symmetry |
| **Applications** | Reactors, heat exchangers | Bioreactors, particles |

### Steady vs. Transient Analysis

| Aspect | Steady-State | Transient |
|--------|--------------|----------|
| **Time dependence** | None | Explicit |
| **Solution method** | Iterative (Gauss-Seidel) | Time-stepping |
| **Convergence** | Error-based | Stability-limited |
| **Applications** | Design, optimization | Process dynamics, safety |
| **Computation** | Moderate iterations | Many time steps |

### Dimensional Analysis - Characteristic Times

**Cartesian (Reactor Block)**:
- Characteristic length: L = 0.2 m
- Thermal diffusivity: α ≈ 1×10⁻⁶ m²/s (typical catalyst bed)
- Thermal time constant: τ = L²/α ≈ 40,000 s ≈ 11 hours
- Steady state reached after 3-5τ ≈ 1-2 days

**Spherical (Bioreactor)**:
- Characteristic length: R = 0.15 m
- Thermal diffusivity: α ≈ 1.5×10⁻⁷ m²/s (water-like)
- Thermal time constant: τ = R²/α ≈ 15,000 s ≈ 4.2 hours
- 95% equilibration: ~1 hour (as observed)

### Practical Considerations

**For Petrochemical Applications** (Steady-State 3D Cartesian):
1. **Reactor Design**: Optimize cooling configuration
2. **Hot Spot Management**: Prevent catalyst deactivation
3. **Selectivity Control**: Temperature affects product distribution
4. **Safety**: Prevent thermal runaway
5. **Scale-up**: Surface-to-volume ratio decreases

**For Biomolecular Applications** (Transient 3D Spherical):
1. **Sterilization**: Ensure all points reach target temperature
2. **Vessel Integrity**: Manage thermal stress
3. **Energy Efficiency**: Minimize heating/cooling times
4. **Product Quality**: Avoid over-processing
5. **Batch Processing**: Optimize cycle times

### Computational Efficiency

**3D Cartesian**: 30³ = 27,000 points
- Memory: ~200 KB (double precision)
- Computation time: ~1-2 minutes for 5000 iterations

**3D Spherical with Symmetry**: 100 points (1D)
- Memory: ~1 KB
- Computation time: ~30 seconds for 100,000 time steps

**Key Lesson**: Exploit symmetry whenever possible!

### Extensions and Advanced Topics

1. **Coupled Phenomena**:
   - Heat and mass transfer (evaporation, reaction)
   - Natural convection (Rayleigh-Bénard)
   - Phase change (melting, freezing)

2. **Variable Properties**:
   - Temperature-dependent thermal conductivity
   - Composition-dependent properties

3. **Complex Geometries**:
   - Finite element methods
   - Body-fitted coordinates
   - Commercial software (COMSOL, ANSYS)

4. **Moving Boundaries**:
   - Stefan problems (melting/freezing)
   - Ablation

5. **Optimization**:
   - Optimal boundary conditions
   - Minimum energy configurations
   - Multi-objective optimization

---
## Exercises

### Exercise 1: Reactor Hot Spot Analysis
For the catalytic reactor block:
- Vary the heat generation rate from 4×10⁵ to 1.2×10⁶ W/m³
- Plot maximum temperature vs. heat generation rate
- Determine the maximum safe heat generation rate if catalyst deactivation occurs above 300°C
- How does the location of the hot spot change?

### Exercise 2: Cooling Optimization
For the reactor block:
- Keep heat generation constant but vary the side cooling temperature
- Compare cooling configurations: (a) all sides at 150°C, (b) bottom/top at 200/180°C, sides at 120°C
- Which configuration minimizes maximum temperature?
- Calculate total heat removal for each case

### Exercise 3: Bioreactor Size Effects
For the spherical bioreactor:
- Compare sterilization times for bioreactor radii of 0.05, 0.10, 0.15, and 0.20 m
- Plot time-to-sterilization vs. radius
- Verify the R² scaling relationship
- Calculate energy requirements for each size

### Exercise 4: Ramped Temperature Sterilization
Modify the bioreactor problem:
- Instead of instantaneous surface temperature change, use a linear ramp: T(t) = T_init + (T_steam - T_init) × (t/t_ramp)
- Use t_ramp = 5, 10, and 15 minutes
- Compare thermal stress (temperature gradient at surface) for each case
- Determine optimal ramp rate

### Exercise 5: Non-Uniform Heat Generation
For the reactor block:
- Implement position-dependent heat generation: q(z) = q₀ × (1 + 0.5 × z/L)
- This represents higher conversion (more heat generation) as reactants flow upward
- Compare temperature distribution with uniform heat generation case
- How does the hot spot location change?

### Exercise 6: Cylindrical vs. Spherical Bioreactor
- Implement a cylindrical bioreactor (r,z) with same volume as the spherical one
- Compare sterilization times
- Which geometry is more efficient for sterilization?
- Consider practical advantages/disadvantages of each shape

### Exercise 7: Transient Reactor Startup
Convert the reactor block to transient:
- Start with uniform temperature (150°C)
- Suddenly turn on heat generation
- Apply boundary conditions as in steady case
- How long to reach steady state?
- Plot maximum temperature vs. time (safety consideration)

### Exercise 8: Multi-Layer Sphere
Extend the bioreactor problem:
- Add a stainless steel wall (0.5 cm thick) outside the liquid
- Use k_steel = 16 W/m·K, ρ_steel = 8000 kg/m³, cp_steel = 500 J/kg·K
- Implement two-region solution with interface continuity
- How does the wall affect sterilization time?

### Challenge Exercise: Optimization Problem
For the reactor block:
- Objective: Minimize maximum temperature
- Design variables: Individual temperatures on all 6 faces (within 100-250°C)
- Constraint: Total heat removal must equal heat generation
- Use optimization algorithm (scipy.optimize) to find optimal boundary conditions
- Bonus: Minimize cost function that weighs both max temperature and total surface area below 200°C